# Support Chatbot con RAG
**Proyecto Integrador — AI Engineering**  
Alumno: Diego Lopez Castan

Este notebook muestra el pipeline RAG completo:
1. Carga de librerías y configuración
2. Carga y parseo del documento (chunks)
3. Generación de embeddings
4. Indexación en ChromaDB
5. Consulta por similitud (k-NN)
6. Generación de respuesta con LLM

## 1. Carga de librerías

In [1]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.schema import Document

# Cargar variables de entorno desde .env
load_dotenv(find_dotenv())

OPENAI_API_KEY   = os.getenv("OPENAI_API_KEY")
EMBEDDING_MODEL  = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")
CHAT_MODEL       = os.getenv("CHAT_MODEL", "gpt-4.1-mini")

print("✅ Librerías cargadas correctamente")
print(f"   Embedding model : {EMBEDDING_MODEL}")
print(f"   Chat model      : {CHAT_MODEL}")

/Users/dlopez/Documents/Apps/Support_chatbot_RAG/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


✅ Librerías cargadas correctamente
   Embedding model : text-embedding-3-small
   Chat model      : gpt-4.1-mini


## 2. Carga y parseo del documento (chunks)

Simulamos un documento de FAQs dividiéndolo en chunks.  
En el proyecto real, `build_index.py` carga los documentos desde disco.

In [2]:
def load_and_chunk_document(raw_text: str, chunk_size: int = 200) -> list[Document]:
    """
    Divide el texto en chunks de tamaño aproximado `chunk_size` caracteres.
    Retorna una lista de objetos Document de LangChain.
    """
    words = raw_text.split()
    chunks = []
    current_chunk = []
    current_len = 0

    for word in words:
        current_chunk.append(word)
        current_len += len(word) + 1
        if current_len >= chunk_size:
            chunks.append(Document(page_content=" ".join(current_chunk)))
            current_chunk = []
            current_len = 0

    if current_chunk:
        chunks.append(Document(page_content=" ".join(current_chunk)))

    return chunks


# Documento de ejemplo (FAQs de soporte)
raw_document = """
¿Cómo restablecer mi contraseña? Podés restablecer tu contraseña desde la página de inicio de sesión usando el enlace de recuperación. Recibirás un email con instrucciones.
¿Qué conductas están prohibidas? Está prohibido usar la plataforma para actividades ilegales, fraudulentas o que vulneren derechos de terceros. También se prohíbe el spam.
¿Cómo optimizar el uso de la plataforma? Se recomienda mantener actualizados los datos, revisar las configuraciones periódicamente y utilizar contraseñas seguras.
¿Qué hacer ante errores frecuentes? Consultá la sección de ayuda antes de contactar soporte. Puede ahorrar tiempo y ofrecer soluciones inmediatas.
¿Cómo cancelar mi suscripción? Podés cancelar tu suscripción desde la sección de configuración de cuenta, en la opción 'Facturación y planes'.
"""

chunks = load_and_chunk_document(raw_document)

print(f"✅ Documento parseado en {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\n  Chunk {i+1}: {chunk.page_content[:80]}...")

✅ Documento parseado en 4 chunks

  Chunk 1: ¿Cómo restablecer mi contraseña? Podés restablecer tu contraseña desde la página...

  Chunk 2: Está prohibido usar la plataforma para actividades ilegales, fraudulentas o que ...

  Chunk 3: actualizados los datos, revisar las configuraciones periódicamente y utilizar co...

  Chunk 4: tiempo y ofrecer soluciones inmediatas. ¿Cómo cancelar mi suscripción? Podés can...


## 3. Generación de embeddings e indexación en ChromaDB

Se usa `text-embedding-3-small` de OpenAI.  
La métrica de similitud es **coseno** (por defecto en Chroma con embeddings de OpenAI).

In [3]:
def generate_embeddings(embedding_model: str) -> OpenAIEmbeddings:
    """
    Inicializa el modelo de embeddings de OpenAI.
    Retorna un objeto OpenAIEmbeddings listo para usar con ChromaDB.
    """
    return OpenAIEmbeddings(model=embedding_model)


def build_vector_store(chunks: list[Document], embeddings: OpenAIEmbeddings) -> Chroma:
    """
    Indexa los chunks en ChromaDB usando los embeddings generados.
    Retorna el vector store listo para consultas.
    """
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name="faqs_notebook"
    )
    return vector_db


embeddings  = generate_embeddings(EMBEDDING_MODEL)
vector_db   = build_vector_store(chunks, embeddings)

print("✅ Embeddings generados e indexados en ChromaDB")
print(f"   Total chunks indexados: {len(chunks)}")

✅ Embeddings generados e indexados en ChromaDB
   Total chunks indexados: 4


## 4. Consulta por similitud k-NN

Se recuperan los `k=3` chunks más similares a la pregunta usando **similitud coseno**.  
Se aplica un umbral de score ≤ 0.5 para filtrar chunks poco relevantes y reducir ruido.

In [4]:
def search_similar_chunks(vector_db: Chroma, question: str, k: int = 3, score_threshold: float = 0.5) -> tuple[list, str]:
    """
    Búsqueda k-NN con similitud coseno sobre ChromaDB.
    Filtra chunks con distancia > score_threshold para minimizar ruido.
    Retorna los chunks filtrados y el contexto concatenado.
    """
    results = vector_db.similarity_search_with_score(question, k=k)
    filtered = [(doc, score) for doc, score in results if score <= score_threshold]

    if not filtered:
        print("⚠️  No se encontraron chunks relevantes con el umbral aplicado.")
        return [], ""

    context = "\n\n".join([doc.page_content for doc, _ in filtered])
    return filtered, context


# Pregunta de prueba
pregunta = "¿Cómo puedo restablecer mi contraseña?"

chunks_retrieved, context = search_similar_chunks(vector_db, pregunta)

print(f"✅ Pregunta: {pregunta}")
print(f"   Chunks recuperados: {len(chunks_retrieved)}")
for i, (doc, score) in enumerate(chunks_retrieved):
    print(f"\n  Chunk {i+1} (score: {score:.4f}): {doc.page_content[:100]}...")

✅ Pregunta: ¿Cómo puedo restablecer mi contraseña?
   Chunks recuperados: 1

  Chunk 1 (score: 0.4835): ¿Cómo restablecer mi contraseña? Podés restablecer tu contraseña desde la página de inicio de sesión...


## 5. Generación de respuesta con LLM

El contexto recuperado se inyecta en el prompt.  
El LLM responde **únicamente** basándose en ese contexto (RAG).

In [5]:
def generate_answer(question: str, context: str, chat_model: str) -> dict:
    """
    Genera una respuesta JSON usando el LLM condicionada al contexto recuperado.
    Retorna un diccionario con 'answer' y 'actions'.
    """
    if not context:
        return {"answer": "No hay información suficiente en el contexto.", "actions": []}

    llm = ChatOpenAI(model=chat_model, temperature=0)

    prompt = f"""Eres un asistente de soporte al cliente.
Responde ÚNICAMENTE usando la información del contexto.
Devuelve SOLO un JSON válido con las claves 'answer' y 'actions' (lista de acciones sugeridas).

Contexto:
{context}

Pregunta:
{question}
"""
    response = llm.invoke(prompt)
    raw = response.content.strip().replace("```json", "").replace("```", "")
    return json.loads(raw)


answer = generate_answer(pregunta, context, CHAT_MODEL)

print("✅ Respuesta generada por el LLM:")
print(json.dumps(answer, indent=2, ensure_ascii=False))

✅ Respuesta generada por el LLM:
{
  "answer": "Podés restablecer tu contraseña desde la página de inicio de sesión usando el enlace de recuperación. Recibirás un email con instrucciones.",
  "actions": [
    "Ir a la página de inicio de sesión",
    "Hacer clic en el enlace de recuperación de contraseña",
    "Revisar el email para seguir las instrucciones"
  ]
}
